In [ ]:
import torch
import pandas as pd

class OnlineStateManager: 
    def __init__(self, num_nodes, num_communities, device='cpu'):
        self.num_nodes = num_nodes
        self.num_communities = num_communities
        self.device = device
        # Static parameters
        self.global_degree = torch.zeros(num_nodes, device=device, dtype=torch.float32)
        # Initialize dynamic states
        self.node_lifespans = torch.zeros(num_nodes, device=device, dtype=torch.float32)
        self.total_duration = 1.0
        self.m = 0.0  # total number of links
        self.T_context = None
        self.last_seen = torch.full((num_nodes,), -1.0, device=device)
        self.last_prob = torch.ones(num_nodes, num_communities, device=device) / num_communities


    def init_context(self):
        """
        [Step 1] 在 pre_scan 后，训练开始前调用。
        使用生命周期初始化 T_Context。
        """
        # 初始化假设：节点把它的生命周期平均分配给所有社区
        # Shape: [N] -> [N, 1] -> [N, K]
        avg_duration = self.node_lifespans.unsqueeze(1) / self.num_communities
        self.T_context = avg_duration.repeat(1, self.num_communities)
        self.last_seen.fill_(-1.0) # 保持是一维
        self.last_prob.fill_(1.0 / self.num_communities)
    

    def get_context_for_loss(self, node_ids):
        """
        [Read] 在计算 Loss 时调用
        """
        # 1. 获取 T_context 所在的设备 (可能是 CPU)
        target_device = self.T_context.device
        
        # 2. 将索引 (node_ids) 挪到同一个设备上
        # 如果 node_ids 已经在该设备，这步操作是零开销的
        ids_on_target = node_ids.to(target_device)
        
        # 3. 进行索引取值
        values = self.T_context[ids_on_target]
        
        # 4. 将取出的结果挪回 node_ids 原来的设备 (通常是 GPU，为了后续算 Loss)
        return values.to(node_ids.device)
    

    def get_last_prob(self, node_ids):
        """
        安全读取上一次的概率分布，自动处理 CPU/GPU 索引冲突
        """
        # 1. 获取存储位置 (CPU)
        target_device = self.last_prob.device 
        # 2. 搬运索引
        ids_on_target = node_ids.to(target_device)
        # 3. 取值
        values = self.last_prob[ids_on_target]
        # 4. 搬回原设备 (GPU)
        return values.to(node_ids.device)

    # [新增] 安全读取 global_degree
    def get_node_degree(self, node_ids):
        """
        安全读取节点度数，自动处理 CPU/GPU 索引冲突
        """
        target_device = self.global_degree.device
        ids_on_target = node_ids.to(target_device)
        values = self.global_degree[ids_on_target]
        return values.to(node_ids.device)
    

    def update_step(self, node_ids, new_probs, current_time):
        """
        [Write] 在 Optimizer.step() 之后调用
        逻辑：Sample-and-Hold 差分累积
        """
        node_ids = node_ids.to(self.device)
        new_probs = new_probs.detach().to(self.device)
        current_time = current_time.to(self.device)
        
        # 1. 获取上次状态
        prev_times = self.last_seen[node_ids]
        prev_probs = self.last_prob[node_ids]
        
        # 2. 识别非首次出现的节点
        # 只有之前出现过 (prev_time != -1)，才有"空窗期"需要填补
        mask = (prev_times != -1)
        
        if mask.any():
            valid_nodes = node_ids[mask]
            
            # 计算时间差 delta_t
            delta_t = current_time[mask] - prev_times[mask]
            # 确保时间非负 (防止数据乱序)
            delta_t = delta_t.clamp(min=0)
            
            # 计算时长增量: 旧分布 * 时间差
            # [M, 1] * [M, K]
            duration_inc = delta_t.unsqueeze(1) * prev_probs[mask]
            
            # 累加到 Buffer
            # 使用 index_add_ 处理 batch 内可能的重复节点
            self.T_context.index_add_(0, valid_nodes, duration_inc)

        # 3. 无论是否首次出现，都更新状态为当前值
        self.last_seen[node_ids] = current_time
        self.last_prob[node_ids] = new_probs